In [ ]:
# ============================================================
# Notebook 06 - Ablation Study
# CS-419 Deep Learning Project - Speech Emotion Recognition
# ============================================================
# Ablation: remove one component at a time from the best model
# and measure the drop in performance.
#
# Components tested:
#   1. Remove Batch Normalization
#   2. Remove Dropout
#   3. Remove L2 regularization
#   4. Remove class weights
#   5. Remove data augmentation
#   6. Replace BiLSTM with simple LSTM
#   7. Remove the LSTM entirely (CNN only)
#
# Results visualized as a delta-accuracy table.
# ============================================================

import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

sys.path.append("../src")

from models import build_cnn_lstm, build_cnn
from train import train_model
from evaluate import evaluate_model
from data_loader import get_class_weights, load_tess_paths, split_dataset
from utils import set_seeds, save_results_csv
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

set_seeds(42)
os.makedirs("results/plots", exist_ok=True)

# ---- Cell 1: Load data ----

train_data = np.load("results/cache/train_spec.npz")
val_data   = np.load("results/cache/val_spec.npz")
test_data  = np.load("results/cache/test_spec.npz")

X_train, y_train = train_data["X"], train_data["y"]
X_val,   y_val   = val_data["X"],   val_data["y"]
X_test,  y_test  = test_data["X"],  test_data["y"]

INPUT_SHAPE = X_train.shape[1:]

DATA_ROOT = "data/TESS Toronto emotional speech set data"
df = load_tess_paths(DATA_ROOT)
df_train, _, _ = split_dataset(df)
class_weights = get_class_weights(df_train)

# Create augmented training data
from augmentation import frequency_mask, time_mask

def augment_batch(X, p=0.6):
    X_aug = X.copy()
    for i in range(len(X_aug)):
        if np.random.rand() < p:
            X_aug[i] = frequency_mask(X_aug[i], max_width=12)
        if np.random.rand() < p:
            X_aug[i] = time_mask(X_aug[i], max_width=15)
    return X_aug

X_aug = augment_batch(X_train, p=0.6)
X_train_aug = np.concatenate([X_train, X_aug])
y_train_aug = np.concatenate([y_train, y_train])
idx = np.random.permutation(len(X_train_aug))
X_train_aug, y_train_aug = X_train_aug[idx], y_train_aug[idx]

# ---- Cell 2: Full model (baseline for ablation) ----

print("=== Full Model (all components) ===")
full_model = build_cnn_lstm(INPUT_SHAPE, cnn_filters=(32, 64, 128),
                             lstm_units=128, dropout_rate=0.4, l2_reg=1e-4)
full_hist = train_model(full_model, X_train_aug, y_train_aug, X_val, y_val,
                         model_name="Ablation_Full", epochs=60, batch_size=32,
                         class_weights=class_weights)
res_full = evaluate_model(full_model, X_test, y_test, model_name="Full_Model")
baseline_acc = res_full["accuracy"]
print(f"Full model accuracy: {baseline_acc*100:.2f}%")

# ---- Cell 3: Helper to build ablated models ----

def build_ablated_model(name, input_shape):
    """Build CNN-LSTM variants with one component removed."""
    inp = keras.Input(shape=input_shape)
    x = inp

    use_bn       = "no_bn"       not in name
    use_dropout  = "no_drop"     not in name
    use_l2       = "no_l2"       not in name
    use_bilstm   = "simple_lstm" not in name and "no_lstm" not in name
    use_lstm     = "no_lstm"     not in name

    reg = regularizers.l2(1e-4) if use_l2 else None

    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, padding="same",
                           kernel_regularizer=reg)(x)
        if use_bn:
            x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D(pool_size=(2, 1))(x)
        if use_dropout:
            x = layers.Dropout(0.2)(x)

    sh = x.shape
    seq_len  = sh[2]
    feat_dim = sh[1] * sh[3]
    x = layers.Reshape((seq_len, feat_dim))(x)

    if use_lstm:
        if use_bilstm:
            x = layers.Bidirectional(
                layers.LSTM(128, return_sequences=False,
                            dropout=0.3 if use_dropout else 0.0)
            )(x)
        else:
            x = layers.LSTM(128, return_sequences=False,
                             dropout=0.3 if use_dropout else 0.0)(x)
    else:
        x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(256, kernel_regularizer=reg)(x)
    if use_bn:
        x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    if use_dropout:
        x = layers.Dropout(0.4)(x)

    out = layers.Dense(7, activation="softmax")(x)
    model = keras.Model(inp, out, name=name)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

# ---- Cell 4: Run ablation experiments ----

ablation_configs = [
    ("no_bn",       "Remove Batch Normalization",  X_train_aug, y_train_aug, class_weights),
    ("no_drop",     "Remove Dropout",              X_train_aug, y_train_aug, class_weights),
    ("no_l2",       "Remove L2 Regularization",   X_train_aug, y_train_aug, class_weights),
    ("simple_lstm", "Replace BiLSTM with LSTM",   X_train_aug, y_train_aug, class_weights),
    ("no_lstm",     "Remove LSTM (CNN only)",      X_train_aug, y_train_aug, class_weights),
    ("full_noaug",  "Remove Augmentation",         X_train,     y_train,     class_weights),
    ("full_nocw",   "Remove Class Weights",        X_train_aug, y_train_aug, None),
]

ablation_results = [{"config": "Full Model", "accuracy": baseline_acc,
                      "delta": 0.0, "description": "All components included"}]

for cfg_name, description, X_tr, y_tr, cw in ablation_configs:
    print(f"\n--- Ablation: {description} ---")
    model = build_ablated_model(cfg_name, INPUT_SHAPE)
    train_model(model, X_tr, y_tr, X_val, y_val,
                model_name=f"Ablation_{cfg_name}", epochs=50, batch_size=32,
                class_weights=cw)
    res = evaluate_model(model, X_test, y_test, model_name=cfg_name)
    delta = res["accuracy"] - baseline_acc
    ablation_results.append({
        "config":      description,
        "accuracy":    res["accuracy"],
        "delta":       delta,
        "description": description,
    })
    print(f"  Accuracy: {res['accuracy']*100:.2f}%  (delta: {delta*100:+.2f}%)")

# ---- Cell 5: Ablation table and chart ----

df_ablation = pd.DataFrame(ablation_results)
save_results_csv(ablation_results, "results/ablation_results.csv")

print("\n=== Ablation Study Results ===")
print(df_ablation[["config", "accuracy", "delta"]].to_string(index=False))

# Horizontal bar chart
fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#4C72B0" if d == 0 else ("#DD8452" if d < 0 else "#55A868")
          for d in df_ablation["delta"]]
bars = ax.barh(df_ablation["config"], df_ablation["delta"] * 100,
               color=colors, edgecolor="white")

ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Change in Test Accuracy (%) vs Full Model")
ax.set_title("Ablation Study - Impact of Each Component",
             fontsize=13, fontweight="bold")
ax.grid(axis="x", alpha=0.3)

for bar, val in zip(bars, df_ablation["delta"] * 100):
    x_pos = val + (0.3 if val >= 0 else -0.3)
    ha    = "left" if val >= 0 else "right"
    ax.text(x_pos, bar.get_y() + bar.get_height() / 2,
            f"{val:+.2f}%", va="center", ha=ha, fontsize=9)

patch_neg  = mpatches.Patch(color="#DD8452", label="Hurts performance")
patch_base = mpatches.Patch(color="#4C72B0", label="Full model baseline")
ax.legend(handles=[patch_base, patch_neg], loc="lower right")

plt.tight_layout()
plt.savefig("results/plots/ablation_study.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n=== Ablation Study Complete ===")
print("Refer to the chart to see which components matter most.")